# 06: Report Generation — Complete Chart Gallery

This notebook demonstrates every chart type available in `siege_utilities.reporting.ChartGenerator`,
plus the `ChartTypeRegistry` extension system; polling / survey analysis now
lives in the dedicated `siege_utilities.survey` pipeline (see notebook 28).

**Sections:**

1. [Imports & Setup](#1-imports--setup)
2. [Standard Charts](#2-standard-charts) — bar, line, pie, scatter, heatmap
3. [Geographic Maps (Folium)](#3-geographic-maps-folium) — marker, heatmap, cluster, flow
4. [Geographic Maps (Matplotlib)](#4-geographic-maps-matplotlib) — advanced choropleth, bivariate choropleth
5. [3D Visualization](#5-3d-visualization) — 3D scatter/surface
6. [Isochrone Service Areas](#5b-isochrone-service-areas) — open-source provider integration
7. [Text-Based Charts](#6-text-based-charts) — proportional bar, heatmap text
8. [Composite Charts & Layout](#7-composite-charts--layout) — dashboard, summary, custom, dispatcher
9. [Complete PDF Report](#8-complete-pdf-report) — multi-page PDF with chart_with_caption, chart_section
10. [PowerPoint Generation](#9-powerpoint-generation)
11. [Chart Type Registry](#10-chart-type-registry) — extensible chart catalogue, custom types, validation
12. [Polling / Survey analysis](#11-polling--survey-analysis) — migration to `survey.build_chain` + `WaveSet` (see notebook 28)
13. [Vector / Raster Chart Export](#12-vector--raster-chart-export) — SVG, EPS, PDF vector output
14. [IDML InDesign Export](#13-idml-indesign-export) — programmatic InDesign file generation

## Prerequisites

```bash
pip install -e /path/to/siege_utilities[geo]
pip install reportlab python-pptx pillow folium
```


## What this shows
Every `ChartGenerator.create_*` method, plus reports, PDF, and extensible `ChartTypeRegistry`.

## Why it matters
Canonical chart/PDF gallery. Polling / survey analysis has moved to the `survey` pipeline (see `28_Polling_Survey_Analysis.ipynb`).

## Prereqs
- `pip install 'siege-utilities[reporting,pdf]'`

## Next
- `12_PowerPoint_Generation.ipynb` for slide output; `28_Polling_Survey_Analysis.ipynb` for the new survey pipeline.


In [ ]:
# 1. Imports & Setup
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML, Image as IPImage
import io
import base64

import siege_utilities as su
su.configure_shared_logging(level="INFO")

from siege_utilities.reporting.chart_generator import ChartGenerator
from siege_utilities.reporting.image_utils import show_rl_image, save_rl_image
from reportlab.platypus import Image as RLImage

# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_06"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Branding configuration ---
from siege_utilities.reporting.client_branding import ClientBrandingManager
_branding_mgr = ClientBrandingManager()
BRANDING = _branding_mgr.get_client_branding('siege_analytics')
COLORS = BRANDING['colors']

su.log_info(f"Output directory: {OUTPUT_DIR}  (exists={OUTPUT_DIR.exists()})")
su.log_info(f"Branding: {BRANDING['name']} (primary={COLORS['primary']})")

# Screen / notebook display — no size limits
chart_gen = ChartGenerator(branding_config=BRANDING, output_dir=OUTPUT_DIR)

# PDF-safe variant — auto-scales oversized charts to fit letter page frame
# (456pt / 6.33in wide, 636pt / 8.83in tall with 1-inch margins)
chart_gen_pdf = ChartGenerator(
    branding_config=BRANDING, output_dir=OUTPUT_DIR,
    max_chart_width=6.0, max_chart_height=8.5,
)

np.random.seed(42)

su.log_info(f"ChartGenerator initialized with {BRANDING['name']} branding")
su.log_info(f"Available create_* methods: {len([m for m in dir(chart_gen) if m.startswith('create_')])}")
su.log_info(f"PDF chart generator: max_chart_width={chart_gen_pdf.max_chart_width}, max_chart_height={chart_gen_pdf.max_chart_height}")

In [ ]:
# Sample Data — used throughout the notebook

# 1. Sales data (bar, line, pie, dashboard, PDF)
sales_data = pd.DataFrame({
    'Month': ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun'],
    'Revenue': [45000, 52000, 48000, 61000, 55000, 67000],
    'Expenses': [32000, 35000, 33000, 38000, 36000, 40000],
    'Units': [150, 175, 160, 205, 185, 225],
})
sales_data['Profit'] = sales_data['Revenue'] - sales_data['Expenses']

# 2. Regional data (pie, scatter, text charts)
regional_data = pd.DataFrame({
    'Region': ['North', 'South', 'East', 'West', 'Central'],
    'Sales': [125000, 98000, 145000, 112000, 89000],
    'Customers': [450, 380, 520, 410, 340],
    'Avg Order': [278, 258, 279, 273, 262],
})

# 3. Point data — US cities (marker map, heatmap map, cluster map, 3D)
cities = [
    ('New York', 40.71, -74.01), ('Los Angeles', 34.05, -118.24),
    ('Chicago', 41.88, -87.63), ('Houston', 29.76, -95.37),
    ('Phoenix', 33.45, -112.07), ('Philadelphia', 39.95, -75.17),
    ('San Antonio', 29.42, -98.49), ('San Diego', 32.72, -117.16),
    ('Dallas', 32.78, -96.80), ('San Jose', 37.34, -121.89),
    ('Austin', 30.27, -97.74), ('Jacksonville', 30.33, -81.66),
    ('San Francisco', 37.77, -122.42), ('Columbus', 39.96, -82.99),
    ('Indianapolis', 39.77, -86.16), ('Fort Worth', 32.76, -97.33),
    ('Charlotte', 35.23, -80.84), ('Seattle', 47.61, -122.33),
    ('Denver', 39.74, -104.98), ('Nashville', 36.16, -86.78),
]
point_data = pd.DataFrame(cities, columns=['name', 'lat', 'lon'])
point_data['value'] = np.random.randint(100, 5000, size=len(point_data))
point_data['category'] = np.random.choice(['Tech', 'Finance', 'Healthcare'], size=len(point_data))

# 4. Flow data — origin/destination pairs
flow_data = pd.DataFrame({
    'origin_lat':  [40.71, 34.05, 41.88, 29.76, 47.61, 39.74, 37.77, 33.45],
    'origin_lon':  [-74.01, -118.24, -87.63, -95.37, -122.33, -104.98, -122.42, -112.07],
    'dest_lat':    [34.05, 41.88, 29.76, 33.45, 37.77, 40.71, 47.61, 39.74],
    'dest_lon':    [-118.24, -87.63, -95.37, -112.07, -122.42, -74.01, -122.33, -104.98],
    'flow_value':  [1500, 2200, 800, 1100, 1800, 950, 1300, 600],
})

# 5. Correlation data (heatmap, scatter, summary charts)
n = 30
correlation_data = pd.DataFrame({
    'x': np.random.randn(n) * 10 + 50,
    'y': np.random.randn(n) * 15 + 70,
    'z': np.random.randn(n) * 5 + 20,
    'w': np.random.randn(n) * 8 + 40,
    'v': np.random.randn(n) * 12 + 60,
})

# 6. Heatmap text data — quarterly sales by region
heatmap_text_data = pd.DataFrame([
    {'quarter': 'Q1', 'region': 'North', 'sales': 120},
    {'quarter': 'Q1', 'region': 'South', 'sales': 95},
    {'quarter': 'Q1', 'region': 'East', 'sales': 140},
    {'quarter': 'Q1', 'region': 'West', 'sales': 110},
    {'quarter': 'Q2', 'region': 'North', 'sales': 135},
    {'quarter': 'Q2', 'region': 'South', 'sales': 105},
    {'quarter': 'Q2', 'region': 'East', 'sales': 155},
    {'quarter': 'Q2', 'region': 'West', 'sales': 125},
    {'quarter': 'Q3', 'region': 'North', 'sales': 150},
    {'quarter': 'Q3', 'region': 'South', 'sales': 115},
    {'quarter': 'Q3', 'region': 'East', 'sales': 165},
    {'quarter': 'Q3', 'region': 'West', 'sales': 130},
    {'quarter': 'Q4', 'region': 'North', 'sales': 160},
    {'quarter': 'Q4', 'region': 'South', 'sales': 125},
    {'quarter': 'Q4', 'region': 'East', 'sales': 180},
    {'quarter': 'Q4', 'region': 'West', 'sales': 145},
])

su.log_info(f"sales_data:       {sales_data.shape}")
su.log_info(f"regional_data:    {regional_data.shape}")
su.log_info(f"point_data:       {point_data.shape}")
su.log_info(f"flow_data:        {flow_data.shape}")
su.log_info(f"correlation_data: {correlation_data.shape}")
su.log_info(f"heatmap_text_data:{heatmap_text_data.shape}")

## 2. Standard Charts

Matplotlib-backed charts that return ReportLab `Image` objects.  We use `show_rl_image()` to
decode the base64 data URI for notebook display.

In [ ]:
# 2a. Bar Chart — vertical
img = chart_gen.create_bar_chart(
    sales_data, x_column='Month', y_column='Revenue',
    title='Monthly Revenue',
)
save_rl_image(img, OUTPUT_DIR / 'bar_vertical.png')
display(show_rl_image(img))

# 2a. Bar Chart — horizontal
img_h = chart_gen.create_bar_chart(
    sales_data, x_column='Month', y_column='Revenue',
    title='Monthly Revenue (Horizontal)',
    chart_type='horizontal',
)
save_rl_image(img_h, OUTPUT_DIR / 'bar_horizontal.png')
display(show_rl_image(img_h))

In [ ]:
# 2b. Line Chart — multi-series
img = chart_gen.create_line_chart(
    sales_data, x_column='Month',
    y_columns=['Revenue', 'Expenses'],
    title='Revenue vs Expenses',
)
save_rl_image(img, OUTPUT_DIR / 'line_revenue_expenses.png')
display(show_rl_image(img))

In [ ]:
# 2c. Pie Chart — with legend table
img = chart_gen.create_pie_chart(
    regional_data,
    labels_column='Region', values_column='Sales',
    title='Sales by Region',
)
save_rl_image(img, OUTPUT_DIR / 'pie_regional_sales.png')
display(show_rl_image(img))

In [ ]:
# 2d. Scatter Plot — with color coding
img = chart_gen.create_scatter_plot(
    correlation_data,
    x_column='x', y_column='y', color_column='z',
    title='Scatter with Color Coding',
)
save_rl_image(img, OUTPUT_DIR / 'scatter_color_coded.png')
display(show_rl_image(img))

In [ ]:
# 2e. Heatmap — correlation matrix
img = chart_gen.create_heatmap(
    correlation_data,
    title='Correlation Matrix',
)
save_rl_image(img, OUTPUT_DIR / 'heatmap_correlation.png')
display(show_rl_image(img))

## 3. Geographic Maps (Folium)

Folium-based methods save HTML files and return placeholder ReportLab images.
We call the ChartGenerator API to exercise it, then also create Folium map objects
directly so we get interactive output in the notebook.

In [ ]:
# 3a. Marker Map
import folium
from folium import plugins

# Call ChartGenerator API (saves HTML, returns placeholder)
placeholder = chart_gen.create_marker_map(
    point_data, latitude_column='lat', longitude_column='lon',
    value_column='value', label_column='name',
    title='US Cities - Marker Map', zoom_level=4,
)
su.log_info("create_marker_map() returned placeholder image")

# Display interactive Folium map inline
m = folium.Map(location=[39.0, -98.0], zoom_start=4)
for _, row in point_data.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=max(5, row['value'] / 200),
        popup=f"<b>{row['name']}</b><br>Value: {row['value']:,}",
        color='red', fill=True, fill_opacity=0.7,
    ).add_to(m)
display(m)

In [ ]:
# 3b. Heatmap Map
placeholder = chart_gen.create_heatmap_map(
    point_data, latitude_column='lat', longitude_column='lon',
    value_column='value',
    title='US Cities - Heatmap',
)
su.log_info("create_heatmap_map() returned placeholder image")

# Display interactive heatmap
m = folium.Map(location=[39.0, -98.0], zoom_start=4, tiles='cartodbpositron')
heat_data = [[row['lat'], row['lon'], row['value']] for _, row in point_data.iterrows()]
plugins.HeatMap(heat_data, radius=25, blur=15).add_to(m)
display(m)

In [ ]:
# 3c. Cluster Map
placeholder = chart_gen.create_cluster_map(
    point_data, latitude_column='lat', longitude_column='lon',
    label_column='name',
    title='US Cities - Cluster Map',
)
su.log_info("create_cluster_map() returned placeholder image")

# Display interactive cluster map
m = folium.Map(location=[39.0, -98.0], zoom_start=4, tiles='cartodbpositron')
mc = plugins.MarkerCluster()
for _, row in point_data.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=f"<b>{row['name']}</b><br>Category: {row['category']}",
    ).add_to(mc)
mc.add_to(m)
display(m)

In [ ]:
# 3d. Flow Map
placeholder = chart_gen.create_flow_map(
    flow_data,
    origin_lat_column='origin_lat', origin_lon_column='origin_lon',
    dest_lat_column='dest_lat', dest_lon_column='dest_lon',
    flow_value_column='flow_value',
    title='US City Flows',
)
su.log_info("create_flow_map() returned placeholder image")

# Display interactive flow map
m = folium.Map(location=[39.0, -98.0], zoom_start=4, tiles='cartodbpositron')
max_flow = flow_data['flow_value'].max()
for _, row in flow_data.iterrows():
    weight = max(1, int(row['flow_value'] / max_flow * 8))
    folium.PolyLine(
        locations=[
            [row['origin_lat'], row['origin_lon']],
            [row['dest_lat'], row['dest_lon']],
        ],
        weight=weight, color='red', opacity=0.6,
        popup=f"Flow: {row['flow_value']:,}",
    ).add_to(m)
    folium.CircleMarker([row['origin_lat'], row['origin_lon']], radius=4, color='blue', fill=True).add_to(m)
    folium.CircleMarker([row['dest_lat'], row['dest_lon']], radius=4, color='green', fill=True).add_to(m)
display(m)

## 4. Geographic Maps (Matplotlib & Folium Choropleth)

These methods require a GeoDataFrame with geometry. We load California counties using the
same `get_census_boundaries()` pattern from `spatial/03_choropleth_maps.ipynb`, then use them for both Folium choropleth
maps and matplotlib-rendered maps.

In [ ]:
# Load CA county boundaries
# get_census_boundaries() caches files via its own internal mechanism.
from siege_utilities.geo import get_census_boundaries

counties = get_census_boundaries(year=2020, geographic_level='county', state_fips='06')

if counties is None:
    su.log_warning(f"get_census_boundaries() returned None.")
    su.log_warning("Retrying with year=2021...")
    counties = get_census_boundaries(year=2021, geographic_level='county', state_fips='06')

if counties is None:
    raise RuntimeError(
        f"Could not download Census county boundaries.\n"
        f"  Check network connectivity and try again, or run NB05 first to cache the shapefile."
    )

ca_counties = counties[counties['statefp'] == '06'].copy()

# Simulate data columns
ca_counties['population'] = np.random.randint(10000, 10000000, size=len(ca_counties))
ca_counties['median_income'] = np.random.randint(40000, 150000, size=len(ca_counties))

su.log_info(f"Loaded {len(ca_counties)} California counties")
su.log_info(f"Columns: {list(ca_counties.columns)[:8]}...")

In [ ]:
# 4a. Choropleth Map (Folium) — uses get_census_boundaries() GeoDataFrame
# Prepare data for Folium choropleth — needs a plain DataFrame + separate GeoJSON
choropleth_data = ca_counties[['geoid', 'population']].copy()

placeholder = chart_gen.create_choropleth_map(
    choropleth_data, geo_data=ca_counties,
    location_column='geoid', value_column='population',
    title='CA Population Choropleth (Folium)',
    map_type='us',
    key_on='feature.properties.geoid',
    fill_color='YlOrRd',
)
su.log_info("create_choropleth_map() — HTML saved, placeholder returned")

# Display interactive Folium map inline
import json
m = folium.Map(location=[37.0, -120.0], zoom_start=6, tiles='cartodbpositron')
folium.Choropleth(
    geo_data=json.loads(ca_counties.to_json()),
    data=choropleth_data,
    columns=['geoid', 'population'],
    key_on='feature.properties.geoid',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name='Population',
).add_to(m)
display(m)

In [ ]:
# 4b. Bivariate Choropleth (Folium) — interactive version of the matplotlib bivariate
placeholder = chart_gen.create_bivariate_choropleth(
    ca_counties, geo_data=ca_counties,
    location_column='geoid',
    value_column1='population', value_column2='median_income',
    title='Population vs Income (Folium)',
    color_scheme='default',
)
su.log_info("create_bivariate_choropleth() — HTML saved, placeholder returned")

# The saved HTML at OUTPUT_DIR/temp_bivariate_choropleth.html is interactive.
# Display the saved map inline:
from IPython.display import IFrame
bivar_path = OUTPUT_DIR / "temp_bivariate_choropleth.html"
if bivar_path.exists():
    display(IFrame(src=str(bivar_path), width=800, height=500))

In [ ]:
# 4c. Advanced Choropleth (matplotlib)
img = chart_gen.create_advanced_choropleth(
    ca_counties, geodata=ca_counties,
    location_column='geoid', value_column='population',
    title='CA Population by County (Quantiles)',
    classification='quantiles', bins=5,
)
save_rl_image(img, OUTPUT_DIR / 'choropleth_quantiles.png')
display(show_rl_image(img))

In [ ]:
# 4d. Bivariate Choropleth (matplotlib)
img = chart_gen.create_bivariate_choropleth_matplotlib(
    ca_counties, geodata=ca_counties,
    location_column='geoid',
    value_column1='population', value_column2='median_income',
    title='Population vs Income (matplotlib)',
)
save_rl_image(img, OUTPUT_DIR / 'bivariate_matplotlib.png')
display(show_rl_image(img))

## 5. 3D Visualization

Uses matplotlib's 3D projection for point-cloud and surface plots.

In [ ]:
# 5a. 3D Map — point cloud with simulated elevation
img = chart_gen.create_3d_map(
    point_data,
    latitude_column='lat', longitude_column='lon',
    elevation_column='value',
    title='3D US City Point Cloud',
)
save_rl_image(img, OUTPUT_DIR / '3d_point_cloud.png')
display(show_rl_image(img))

## 5b. Isochrone Service Areas

This demonstrates the open-source isochrone API integration (ORS/Valhalla), including custom server support.


In [ ]:
import os
import json
from siege_utilities.geo.isochrones import build_isochrone_request, get_isochrone

# Example center: Chicago
center_lat, center_lon = 41.8781, -87.6298

# Build request definitions without making network calls
ors_req = build_isochrone_request(
    latitude=center_lat,
    longitude=center_lon,
    travel_time_minutes=15,
    provider='openrouteservice',
    profile='driving-car',
)

valhalla_req = build_isochrone_request(
    latitude=center_lat,
    longitude=center_lon,
    travel_time_minutes=20,
    provider='valhalla',
    base_url='http://valhalla.svc.cluster.local:8002',
    profile='driving-car',
)

print('ORS endpoint:', ors_req['url'])
print('Valhalla endpoint:', valhalla_req['url'])

# Optional live request when ORS_API_KEY is available
ors_api_key = os.getenv('ORS_API_KEY')
if ors_api_key:
    try:
        iso = get_isochrone(
            latitude=center_lat,
            longitude=center_lon,
            travel_time_minutes=15,
            provider='openrouteservice',
            api_key=ors_api_key,
            profile='driving-car',
        )
        out_geojson = OUTPUT_DIR / 'isochrone_chicago_15min.geojson'
        with open(out_geojson, 'w', encoding='utf-8') as f:
            json.dump(iso, f)
        print('Saved:', out_geojson)
    except Exception as e:
        print('Live ORS isochrone request failed:', type(e).__name__, str(e))
else:
    print('ORS_API_KEY not set; skipping live isochrone download.')


## 6. Text-Based Charts

These return HTML strings instead of images — useful for email reports, lightweight dashboards,
or environments without matplotlib.

In [ ]:
# 6a. Proportional Text Bar Chart
text_chart = chart_gen.create_proportional_text_bar_chart(
    regional_data,
    labels_column='Region', values_column='Sales',
    title='Sales by Region (Text)',
)
display(HTML(f'<pre style="font-family: monospace; line-height: 1.4;">{text_chart}</pre>'))

In [ ]:
# 6b. Heatmap Text Chart
text_heatmap = chart_gen.create_heatmap_text_chart(
    heatmap_text_data,
    x_column='quarter', y_column='region', value_column='sales',
    title='Quarterly Sales by Region (Text)',
)
display(HTML(f'<pre style="font-family: monospace; line-height: 1.4;">{text_heatmap}</pre>'))

## 7. Composite Charts & Layout

Higher-level methods that combine multiple charts or dispatch by configuration.

In [ ]:
# 7a. Dashboard — 2x2 grid of four chart types
months = sales_data['Month'].tolist()
revenue = sales_data['Revenue'].tolist()
expenses = sales_data['Expenses'].tolist()

dashboard_charts = [
    {
        'type': 'bar',
        'title': 'Revenue by Month',
        'data': {'labels': months, 'datasets': [{'data': revenue}]},
    },
    {
        'type': 'line',
        'title': 'Revenue Trend',
        'data': {'labels': months, 'datasets': [{'data': revenue}]},
    },
    {
        'type': 'pie',
        'title': 'Regional Sales',
        'data': {'labels': regional_data['Region'].tolist(), 'data': regional_data['Sales'].tolist()},
    },
    {
        'type': 'scatter',
        'title': 'X vs Y',
        'data': {'x': correlation_data['x'].tolist(), 'y': correlation_data['y'].tolist()},
    },
]

img = chart_gen.create_dashboard(dashboard_charts, layout='2x2')
save_rl_image(img, OUTPUT_DIR / 'dashboard_2x2.png')
display(show_rl_image(img))

In [ ]:
# 7b. DataFrame Summary Charts — auto-histogram for numeric columns
img = chart_gen.create_dataframe_summary_charts(
    correlation_data,
    title='Data Distributions',
)
save_rl_image(img, OUTPUT_DIR / 'summary_histograms.png')
display(show_rl_image(img))

In [ ]:
# 7c. Custom Chart Dispatcher — create_custom_chart()
config = {
    'type': 'bar',
    'data': sales_data,
    'title': 'Custom Bar Chart via Dispatcher',
}
img = chart_gen.create_custom_chart(config)
save_rl_image(img, OUTPUT_DIR / 'custom_bar.png')
display(show_rl_image(img))

In [ ]:
# 7d. generate_chart_from_dataframe — generic dispatcher
for chart_type in ['bar', 'line', 'pie', 'scatter']:
    su.log_info(f"Generating {chart_type} chart...")
    img = chart_gen.generate_chart_from_dataframe(
        sales_data, chart_type=chart_type,
        x_column='Month', y_columns=['Revenue'],
        title=f'{chart_type.title()} via Dispatcher',
    )
    save_rl_image(img, OUTPUT_DIR / f'dispatch_{chart_type}.png')
    display(show_rl_image(img))

# Heatmap uses correlation_data (all-numeric) — dispatcher computes correlation matrix
su.log_info("Generating heatmap chart...")
img = chart_gen.generate_chart_from_dataframe(
    correlation_data, chart_type='heatmap',
    title='Heatmap via Dispatcher',
)
save_rl_image(img, OUTPUT_DIR / 'dispatch_heatmap.png')
display(show_rl_image(img))

## 8. Complete PDF Report

Build a multi-page PDF demonstrating `BaseReportTemplate`, `create_chart_with_caption()`,
and `create_chart_section()` — these return ReportLab flowable lists that only make sense
inside a PDF build pipeline.

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak, HRFlowable
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from siege_utilities.reporting.templates.base_template import BaseReportTemplate

output_pdf = str(OUTPUT_DIR / 'complete_chart_gallery.pdf')

# --- Branding colors as ReportLab color objects ---
brand_primary = colors.HexColor(COLORS['primary'])       # #1D365D dark navy
brand_secondary = colors.HexColor(COLORS['secondary'])   # #4A90E2 bright blue
brand_accent = colors.HexColor(COLORS['accent'])         # #FF6B35 orange
brand_bg = colors.HexColor(COLORS['background'])         # #F6F6F6 light gray

# PDF chart generator — 200 DPI, sized to fit letter page with headings/captions
chart_gen_pdf = ChartGenerator(
    branding_config=BRANDING, output_dir=OUTPUT_DIR,
    max_chart_width=6.0, max_chart_height=5.5,
)
chart_gen_pdf.default_dpi = 200

template = BaseReportTemplate(
    output_filename=output_pdf,
    page_size='letter',
    margins={'top': 1.0, 'bottom': 1.0, 'left': 1.0, 'right': 1.0},
)

# --- Custom branded styles ---
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(
    'BrandTitle', parent=styles['Title'],
    fontSize=28, textColor=brand_primary, spaceAfter=6,
))
styles.add(ParagraphStyle(
    'BrandSubtitle', parent=styles['Normal'],
    fontSize=14, textColor=brand_secondary, alignment=TA_CENTER, spaceAfter=20,
))
styles.add(ParagraphStyle(
    'BrandHeading', parent=styles['Heading1'],
    fontSize=18, textColor=brand_primary, spaceBefore=0, spaceAfter=8,
))
styles.add(ParagraphStyle(
    'BrandCaption', parent=styles['Normal'],
    fontSize=9, textColor=colors.HexColor('#666666'), italic=True,
    spaceBefore=4, spaceAfter=2,
))
styles.add(ParagraphStyle(
    'BrandMeta', parent=styles['Normal'],
    fontSize=10, textColor=colors.HexColor('#666666'), alignment=TA_CENTER,
))

story = []

# --- Branded title page ---
story.append(Spacer(1, 2.0 * inch))
story.append(HRFlowable(width='80%', thickness=3, color=brand_primary, spaceAfter=12))
story.append(Paragraph('Complete Chart Gallery', styles['BrandTitle']))
story.append(Paragraph('Every ChartGenerator chart type rendered at 200 DPI', styles['BrandSubtitle']))
story.append(HRFlowable(width='80%', thickness=1, color=brand_secondary, spaceBefore=8, spaceAfter=24))
story.append(Paragraph('Prepared by: siege_utilities NB06', styles['BrandMeta']))
story.append(Spacer(1, 6))
story.append(Paragraph('Date: 2026', styles['BrandMeta']))

# Helper: one chart per page with branded heading and caption
def add_chart_page(story, heading, chart_img, caption):
    story.append(PageBreak())
    story.append(HRFlowable(width='100%', thickness=2, color=brand_primary, spaceAfter=8))
    story.append(Paragraph(heading, styles['BrandHeading']))
    story.append(Spacer(1, 8))
    story.extend(chart_gen_pdf.create_chart_with_caption(chart_img, caption))
    story.append(Spacer(1, 6))
    story.append(HRFlowable(width='40%', thickness=0.5, color=brand_secondary))

# =====================================================================
# 1. Bar Chart — Vertical
# =====================================================================
add_chart_page(story, '1a. Bar Chart (Vertical)',
    chart_gen_pdf.create_bar_chart(
        sales_data, x_column='Month', y_column='Revenue',
        title='Monthly Revenue',
    ),
    'Figure 1a: Vertical bar chart — monthly revenue.',
)

# =====================================================================
# 1b. Bar Chart — Horizontal
# =====================================================================
add_chart_page(story, '1b. Bar Chart (Horizontal)',
    chart_gen_pdf.create_bar_chart(
        sales_data, x_column='Month', y_column='Revenue',
        title='Monthly Revenue (Horizontal)', chart_type='horizontal',
    ),
    'Figure 1b: Horizontal bar chart — same data, rotated layout.',
)

# =====================================================================
# 2. Line Chart
# =====================================================================
add_chart_page(story, '2. Line Chart',
    chart_gen_pdf.create_line_chart(
        sales_data, x_column='Month', y_columns=['Revenue', 'Expenses'],
        title='Revenue vs Expenses',
    ),
    'Figure 2: Multi-series line chart comparing revenue and expenses.',
)

# =====================================================================
# 3. Pie Chart
# =====================================================================
add_chart_page(story, '3. Pie Chart',
    chart_gen_pdf.create_pie_chart(
        regional_data, labels_column='Region', values_column='Sales',
        title='Sales by Region',
    ),
    'Figure 3: Pie chart with legend table showing regional sales distribution.',
)

# =====================================================================
# 4. Scatter Plot
# =====================================================================
add_chart_page(story, '4. Scatter Plot',
    chart_gen_pdf.create_scatter_plot(
        correlation_data, x_column='x', y_column='y', color_column='z',
        title='Scatter with Color Coding',
    ),
    'Figure 4: Scatter plot with continuous color coding by z-value.',
)

# =====================================================================
# 5. Heatmap (Correlation Matrix)
# =====================================================================
add_chart_page(story, '5. Correlation Heatmap',
    chart_gen_pdf.create_heatmap(
        correlation_data, title='Correlation Matrix',
    ),
    'Figure 5: Seaborn heatmap showing pairwise correlations.',
)

# =====================================================================
# 6. Advanced Choropleth (matplotlib)
# =====================================================================
add_chart_page(story, '6. Advanced Choropleth',
    chart_gen_pdf.create_advanced_choropleth(
        ca_counties, geodata=ca_counties,
        location_column='geoid', value_column='population',
        title='CA Population by County (Quantiles)',
        classification='quantiles', bins=5,
    ),
    'Figure 6: Quantile-classified choropleth of California county populations.',
)

# =====================================================================
# 7. Bivariate Choropleth (matplotlib)
# =====================================================================
add_chart_page(story, '7. Bivariate Choropleth',
    chart_gen_pdf.create_bivariate_choropleth_matplotlib(
        ca_counties, geodata=ca_counties,
        location_column='geoid',
        value_column1='population', value_column2='median_income',
        title='Population vs Median Income',
    ),
    'Figure 7: Bivariate choropleth — population vs median income with 3x3 color grid.',
)

# =====================================================================
# 8. 3D Point Cloud
# =====================================================================
add_chart_page(story, '8. 3D Visualization',
    chart_gen_pdf.create_3d_map(
        point_data, latitude_column='lat', longitude_column='lon',
        elevation_column='value', title='3D US City Point Cloud',
    ),
    'Figure 8: 3D point cloud — US cities with elevation proportional to value.',
)

# =====================================================================
# 9. Dashboard (2x2)
# =====================================================================
add_chart_page(story, '9. Dashboard',
    chart_gen_pdf.create_dashboard(dashboard_charts, layout='2x2'),
    'Figure 9: Four-panel dashboard — bar, line, pie, and scatter.',
)

# =====================================================================
# 10. Summary Histograms
# =====================================================================
add_chart_page(story, '10. Distribution Summary',
    chart_gen_pdf.create_dataframe_summary_charts(
        correlation_data, title='Data Distributions',
    ),
    'Figure 10: Auto-generated histograms for all numeric columns.',
)

# =====================================================================
# Build PDF
# =====================================================================
template.build_document(story)

import os
pdf_size_kb = os.path.getsize(output_pdf) / 1024
su.log_info(f"PDF written to {output_pdf} ({pdf_size_kb:.0f} KB)")

## 9. PowerPoint Generation

In [ ]:
try:
    from pptx import Presentation
    from pptx.util import Inches, Pt, Emu
    from pptx.dml.color import RGBColor
    from pptx.enum.text import PP_ALIGN
    from datetime import datetime

    prs = Presentation()
    prs.slide_width = Inches(13.333)   # widescreen 16:9
    prs.slide_height = Inches(7.5)

    # Branding colors for PPTX
    pptx_primary = RGBColor(0x1D, 0x36, 0x5D)    # #1D365D
    pptx_secondary = RGBColor(0x4A, 0x90, 0xE2)   # #4A90E2
    pptx_accent = RGBColor(0xFF, 0x6B, 0x35)       # #FF6B35
    pptx_white = RGBColor(0xFF, 0xFF, 0xFF)
    pptx_gray = RGBColor(0x66, 0x66, 0x66)

    # --- Helper: add branded title slide ---
    def add_title_slide(prs, title, subtitle=''):
        slide = prs.slides.add_slide(prs.slide_layouts[6])  # blank layout
        # Navy background
        bg = slide.background
        fill = bg.fill
        fill.solid()
        fill.fore_color.rgb = pptx_primary
        # Title
        txBox = slide.shapes.add_textbox(Inches(1), Inches(2.0), Inches(11.3), Inches(1.5))
        tf = txBox.text_frame
        tf.word_wrap = True
        p = tf.paragraphs[0]
        p.text = title
        p.font.size = Pt(40)
        p.font.bold = True
        p.font.color.rgb = pptx_white
        p.alignment = PP_ALIGN.CENTER
        # Subtitle
        if subtitle:
            p2 = tf.add_paragraph()
            p2.text = subtitle
            p2.font.size = Pt(18)
            p2.font.color.rgb = pptx_secondary
            p2.alignment = PP_ALIGN.CENTER
        # Accent line
        from pptx.shapes.autoshape import Shape
        line = slide.shapes.add_shape(
            1, Inches(3), Inches(3.8), Inches(7.3), Pt(3),  # 1 = rectangle
        )
        line.fill.solid()
        line.fill.fore_color.rgb = pptx_accent
        line.line.fill.background()
        return slide

    # --- Helper: add chart image slide ---
    def add_chart_slide(prs, title, image_path, caption=''):
        slide = prs.slides.add_slide(prs.slide_layouts[6])  # blank
        # Title bar at top
        bar = slide.shapes.add_shape(1, Inches(0), Inches(0), prs.slide_width, Inches(0.9))
        bar.fill.solid()
        bar.fill.fore_color.rgb = pptx_primary
        bar.line.fill.background()
        txBox = slide.shapes.add_textbox(Inches(0.5), Inches(0.1), Inches(12), Inches(0.7))
        tf = txBox.text_frame
        p = tf.paragraphs[0]
        p.text = title
        p.font.size = Pt(24)
        p.font.bold = True
        p.font.color.rgb = pptx_white
        # Chart image — centered
        img_path = str(image_path)
        pic = slide.shapes.add_picture(img_path, Inches(1.5), Inches(1.2), Inches(10.3), Inches(5.5))
        # Caption
        if caption:
            txBox2 = slide.shapes.add_textbox(Inches(1.5), Inches(6.9), Inches(10.3), Inches(0.4))
            tf2 = txBox2.text_frame
            p2 = tf2.paragraphs[0]
            p2.text = caption
            p2.font.size = Pt(10)
            p2.font.italic = True
            p2.font.color.rgb = pptx_gray
            p2.alignment = PP_ALIGN.LEFT
        return slide

    # --- Helper: add data table slide ---
    def add_table_slide(prs, title, df):
        slide = prs.slides.add_slide(prs.slide_layouts[6])  # blank
        # Title bar
        bar = slide.shapes.add_shape(1, Inches(0), Inches(0), prs.slide_width, Inches(0.9))
        bar.fill.solid()
        bar.fill.fore_color.rgb = pptx_primary
        bar.line.fill.background()
        txBox = slide.shapes.add_textbox(Inches(0.5), Inches(0.1), Inches(12), Inches(0.7))
        tf = txBox.text_frame
        p = tf.paragraphs[0]
        p.text = title
        p.font.size = Pt(24)
        p.font.bold = True
        p.font.color.rgb = pptx_white
        # Table
        rows, cols = len(df) + 1, len(df.columns)
        tbl = slide.shapes.add_table(rows, cols, Inches(0.8), Inches(1.3), Inches(11.7), Inches(0.5 * rows)).table
        # Header row
        for i, col_name in enumerate(df.columns):
            cell = tbl.cell(0, i)
            cell.text = str(col_name)
            for paragraph in cell.text_frame.paragraphs:
                paragraph.font.size = Pt(11)
                paragraph.font.bold = True
                paragraph.font.color.rgb = pptx_white
            cell.fill.solid()
            cell.fill.fore_color.rgb = pptx_primary
        # Data rows
        for row_idx, (_, row) in enumerate(df.iterrows()):
            for col_idx, value in enumerate(row):
                cell = tbl.cell(row_idx + 1, col_idx)
                cell.text = str(value)
                for paragraph in cell.text_frame.paragraphs:
                    paragraph.font.size = Pt(10)
                # Alternate row shading
                if row_idx % 2 == 0:
                    cell.fill.solid()
                    cell.fill.fore_color.rgb = RGBColor(0xF6, 0xF6, 0xF6)
        return slide

    # =================================================================
    # Build the deck
    # =================================================================

    # 1. Title slide
    add_title_slide(prs,
        'Complete Chart Gallery',
        f'siege_utilities ChartGenerator | {datetime.now().strftime("%Y-%m-%d")}',
    )

    # 2. Sales data table
    add_table_slide(prs, 'Sales Summary — H1 2026', sales_data)

    # 3-13. Chart slides — one per saved PNG
    chart_slides = [
        ('Bar Chart (Vertical)',     OUTPUT_DIR / 'bar_vertical.png',        'Vertical bar chart — monthly revenue'),
        ('Bar Chart (Horizontal)',   OUTPUT_DIR / 'bar_horizontal.png',      'Horizontal bar chart — same data, rotated layout'),
        ('Line Chart',               OUTPUT_DIR / 'line_revenue_expenses.png','Multi-series line chart — revenue vs expenses'),
        ('Pie Chart',                OUTPUT_DIR / 'pie_regional_sales.png',  'Pie chart with legend table — regional sales'),
        ('Scatter Plot',             OUTPUT_DIR / 'scatter_color_coded.png', 'Scatter plot with color coding by z-value'),
        ('Correlation Heatmap',      OUTPUT_DIR / 'heatmap_correlation.png', 'Seaborn heatmap — pairwise correlations'),
        ('Advanced Choropleth',      OUTPUT_DIR / 'choropleth_quantiles.png','Quantile-classified choropleth — CA counties'),
        ('Bivariate Choropleth',     OUTPUT_DIR / 'bivariate_matplotlib.png','Bivariate choropleth — population vs income'),
        ('3D Visualization',         OUTPUT_DIR / '3d_point_cloud.png',     '3D point cloud — US cities'),
        ('Dashboard (2x2)',          OUTPUT_DIR / 'dashboard_2x2.png',      'Four-panel dashboard — bar, line, pie, scatter'),
        ('Distribution Summary',     OUTPUT_DIR / 'summary_histograms.png', 'Auto-generated histograms — all numeric columns'),
    ]

    added = 0
    for title, img_path, caption in chart_slides:
        if img_path.exists():
            add_chart_slide(prs, title, img_path, caption)
            added += 1
        else:
            su.log_warning(f"Skipping PPTX slide '{title}' — {img_path.name} not found")

    pptx_path = str(OUTPUT_DIR / 'sales_presentation.pptx')
    prs.save(pptx_path)
    pptx_kb = Path(pptx_path).stat().st_size / 1024
    su.log_info(f"PowerPoint created: {pptx_path} ({pptx_kb:.0f} KB, {len(prs.slides)} slides — 1 title + 1 table + {added} charts)")

except ImportError:
    su.log_warning("python-pptx not installed. Run: pip install python-pptx")

## 10. Chart Type Registry

The `ChartTypeRegistry` provides an extensible catalogue of chart configurations.
Each `ChartType` dataclass describes required/optional parameters, rendering defaults,
and feature flags (interactive, 3D, animation). The registry ships with built-in
types for geographic, statistical, temporal, and comparative charts, and supports
custom registrations at runtime.

In [ ]:
# 10. Chart Type Registry
# -----------------------------------------------------------------------
# Demonstrates ChartTypeRegistry: listing built-in chart types, inspecting
# categories, looking up individual types, and registering custom entries.
# -----------------------------------------------------------------------

try:
    from siege_utilities.reporting.chart_types import ChartTypeRegistry, ChartType

    # --- Instantiate a fresh registry (comes pre-loaded with defaults) ---
    registry = ChartTypeRegistry()

    # --- List all registered chart type names ---
    all_types = registry.list_chart_types()
    su.log_info(f"Total registered chart types: {len(all_types)}")
    for name in all_types:
        ct = registry.get_chart_type(name)
        su.log_info(f"  {name:35s}  category={ct.category}")

    # --- List available categories ---
    categories = registry.get_chart_categories()
    su.log_info(f"\nChart categories: {categories}")

    # --- Filter by category ---
    for cat in categories:
        cat_types = registry.list_chart_types(category=cat)
        su.log_info(f"  {cat:15s}: {cat_types}")

    # --- Look up a specific chart type and inspect its help dict ---
    help_info = registry.get_chart_help('bivariate_choropleth')
    su.log_info(f"\nHelp for 'bivariate_choropleth':")
    for key, value in help_info.items():
        su.log_info(f"  {key}: {value}")

    # --- Register a custom chart type ---
    custom_chart = ChartType(
        name='funnel_chart',
        category='comparative',
        description='Funnel chart for conversion pipeline visualization',
        required_parameters=['data', 'stage_column', 'value_column'],
        optional_parameters={
            'title': '',
            'color_scheme': 'blues',
            'show_percentages': True,
        },
        supports_interactive=False,
        supports_3d=False,
        default_width=8.0,
        default_height=6.0,
        custom_options={
            'orientation': 'vertical',
            'gap': 0.02,
        },
    )
    registry.register_chart_type(custom_chart)
    su.log_info(f"\nAfter registration — total chart types: {len(registry.list_chart_types())}")
    su.log_info(f"'funnel_chart' in registry: {registry.get_chart_type('funnel_chart') is not None}")

    # --- Validate parameters for a chart type ---
    valid = registry.validate_chart_parameters(
        'bar_chart',
        data=sales_data, x_column='Month', y_column='Revenue',
    )
    su.log_info(f"\nParameter validation for bar_chart (valid args): {valid}")

    missing_valid = registry.validate_chart_parameters(
        'bar_chart',
        data=sales_data,  # missing x_column, y_column
    )
    su.log_info(f"Parameter validation for bar_chart (missing args): {missing_valid}")

except ImportError as exc:
    su.log_warning(f"ChartTypeRegistry not available: {exc}")

## 11. Polling / Survey analysis

> **Migrated** from `PollingAnalyzer` (deprecated, ELE-2439) to the survey pipeline:
> `siege_utilities.survey.build_chain` → `Chain` → `chain_to_argument` → `Argument`.
> For full end-to-end coverage including multi-wave `WaveSet` comparison and
> trend / heatmap charts, see `notebooks/28_Polling_Survey_Analysis.ipynb`.

The cell below shows a minimal cross-tab using the new primitives — no analyzer
class, no `metric='sessions'` legacy default. Works on any tidy respondent-level
DataFrame.


In [ ]:
# 11. Polling / Survey analysis — migrated from PollingAnalyzer (ELE-2439)
# -----------------------------------------------------------------------
# The cross-tab below uses the survey pipeline primitives instead of the
# deprecated PollingAnalyzer class. See notebooks/28 for multi-wave work.
# -----------------------------------------------------------------------

from siege_utilities.survey import build_chain, chain_to_argument
from siege_utilities.reporting.pages.page_models import TableType

np.random.seed(99)
n_rows = 500
polling_data = pd.DataFrame({
    'channel': np.random.choice(['organic', 'paid', 'direct', 'referral'], n_rows),
    'region':  np.random.choice(['North', 'South', 'East', 'West'], n_rows),
    'value':   np.random.randint(50, 2000, n_rows),
})

chain = build_chain(
    polling_data,
    row_var='channel',
    break_vars=['region'],
    metric='value',
    table_type=TableType.CROSS_TAB,
)
argument = chain_to_argument(
    chain,
    headline='Channel performance by region',
    narrative='Organic leads in the North; paid concentrates in the East.',
)
su.log_info(
    f'built Argument layout={argument.layout!r} table_shape={argument.table.shape}'
)
argument.table


<cell_type>markdown</cell_type>## Method Reference

Complete inventory of `ChartGenerator.create_*` methods and reporting components — all demonstrated above.

| # | Method | Backend | Returns | Section |
|---|--------|---------|---------|--------|
| 1 | `create_bar_chart` | matplotlib | RL Image | 2a |
| 2 | `create_line_chart` | matplotlib | RL Image | 2b |
| 3 | `create_pie_chart` | matplotlib | RL Image | 2c |
| 4 | `create_scatter_plot` | matplotlib | RL Image | 2d |
| 5 | `create_heatmap` | seaborn | RL Image | 2e |
| 6 | `create_choropleth_map` | Folium | Placeholder | 4a |
| 7 | `create_bivariate_choropleth` | Folium + gpd | Placeholder | 4b |
| 8 | `create_bivariate_choropleth_matplotlib` | matplotlib + gpd | RL Image | 4d |
| 9 | `create_advanced_choropleth` | matplotlib + gpd | RL Image | 4c |
| 10 | `create_marker_map` | Folium | Placeholder | 3a |
| 11 | `create_3d_map` | matplotlib 3D | RL Image | 5a |
| 12 | `create_heatmap_map` | Folium | Placeholder | 3b |
| 13 | `create_cluster_map` | Folium | Placeholder | 3c |
| 14 | `create_flow_map` | Folium | Placeholder | 3d |
| 15 | `create_proportional_text_bar_chart` | text | HTML string | 6a |
| 16 | `create_heatmap_text_chart` | text | HTML string | 6b |
| 17 | `create_dashboard` | matplotlib | RL Image | 7a |
| 18 | `create_dataframe_summary_charts` | matplotlib | RL Image | 7b |
| 19 | `create_chart_with_caption` | ReportLab | List[flowable] | 8 |
| 20 | `create_chart_section` | ReportLab | List[flowable] | 8 |
| 21 | `create_custom_chart` | dispatcher | RL Image | 7c |
| -- | `generate_chart_from_dataframe` | dispatcher | RL Image | 7d |

### Chart Type Registry (Section 10)

| Class | Module | Purpose |
|-------|--------|---------|
| `ChartTypeRegistry` | `siege_utilities.reporting.chart_types` | Extensible registry of chart type definitions |
| `ChartType` | `siege_utilities.reporting.chart_types` | Dataclass describing one chart type's params, defaults, and feature flags |

Key methods: `list_chart_types()`, `get_chart_type()`, `register_chart_type()`, `get_chart_categories()`, `get_chart_help()`, `validate_chart_parameters()`, `create_chart()`, `export_chart_type_config()`

### Polling / Survey analysis (Section 11)

| Class / Callable | Module | Purpose |
|-------|--------|---------|
| `build_chain` | `siege_utilities.survey.crosstab` | Build a `Chain` cross-tab from any respondent-level DataFrame |
| `chain_to_argument` | `siege_utilities.survey.render` | Render a `Chain` as a page-ready `Argument` |
| `WaveSet.compare_chain` | `siege_utilities.survey.models` | Longitudinal comparison across waves |
| `trend_chart` / `heatmap` | `siege_utilities.reporting.wave_charts` | Matplotlib figures from a LONGITUDINAL `Chain` |

`PollingAnalyzer` is **deprecated** (ELE-2439); these primitives replace it. End-to-end walkthrough: `notebooks/28_Polling_Survey_Analysis.ipynb`.

**Notes:**
- Methods 19 & 20 (`create_chart_with_caption`, `create_chart_section`) return ReportLab flowable lists — only usable inside a PDF build pipeline (Section 8).

**Files Generated** (all under `OUTPUT_DIR = output/notebook_06/`):
- `bar_vertical.png`, `bar_horizontal.png` — Bar charts
- `line_revenue_expenses.png` — Line chart
- `pie_regional_sales.png` — Pie chart
- `scatter_color_coded.png` — Scatter plot
- `heatmap_correlation.png` — Correlation heatmap
- `choropleth_quantiles.png` — Advanced choropleth (matplotlib)
- `bivariate_matplotlib.png` — Bivariate choropleth (matplotlib)
- `3d_point_cloud.png` — 3D point cloud
- `dashboard_2x2.png` — 2x2 dashboard
- `summary_histograms.png` — Summary histograms
- `custom_bar.png` — Custom dispatcher chart
- `dispatch_bar.png`, `dispatch_line.png`, `dispatch_pie.png`, `dispatch_scatter.png`, `dispatch_heatmap.png` — Dispatched charts
- `complete_chart_gallery.pdf` — Multi-page PDF with captioned charts
- `sales_presentation.pptx` — PowerPoint deck
- Folium HTML maps (marker, heatmap, cluster, flow, choropleth, bivariate)

## 12. Vector / Raster Chart Export

The `ChartGenerator` method `save_figure_as_vector()` exports any matplotlib figure
to **SVG**, **EPS**, or **PDF** format. This is essential for designer handoff —
InDesign and Illustrator can import SVG/EPS directly at any resolution.

### When to use

- **Print reports**: Export SVG/EPS for infinite-resolution print output.
- **Design handoff**: Give designers editable vector charts they can restyle.
- **Archival**: PDF vector copies preserve chart fidelity indefinitely.

In [ ]:
# 12. Vector / Raster Chart Export
# -----------------------------------------------------------------------
# Demonstrates save_figure_as_vector() for SVG, EPS, and PDF output.
# -----------------------------------------------------------------------

import tempfile

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    su.log_info("matplotlib not installed — skipping vector export demo.")

if HAS_MATPLOTLIB:
    from siege_utilities.reporting.chart_generator import ChartGenerator

    cg = ChartGenerator()

    # --- Build a simple chart to export ---
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(["Q1", "Q2", "Q3", "Q4"], [120, 180, 150, 210], color="#4a90d9")
    ax.set_title("Quarterly Revenue")
    ax.set_ylabel("Revenue ($k)")

    with tempfile.TemporaryDirectory() as tmpdir:
        svg_path = cg.save_figure_as_vector(fig, f"{tmpdir}/revenue", fmt="svg")
        eps_path = cg.save_figure_as_vector(fig, f"{tmpdir}/revenue", fmt="eps")
        pdf_path = cg.save_figure_as_vector(fig, f"{tmpdir}/revenue", fmt="pdf")

        su.log_info("Exported vector files:")
        for p in [svg_path, eps_path, pdf_path]:
            if p and p.exists():
                su.log_info(f"  {p.name:<20} {p.stat().st_size:>8,} bytes")

    plt.close(fig)
    su.log_info("Vector export complete.")

## 13. IDML InDesign Export

The `reporting.idml_export` module enables programmatic creation of Adobe InDesign
IDML files. IDML is a ZIP archive containing XML files that InDesign can open directly.

### Two modes

1. **Template mode** (requires `simpleidml`): Load an `.idml` template and replace placeholders.
2. **Manual mode** (zero dependencies): Build the IDML ZIP from scratch.

### When to use

- Producing InDesign-ready reports from analytics pipelines.
- Letting designers customise report layouts while preserving data-driven content.
- Generating multi-format output (PDF + PPTX + IDML) from a single data source.

In [ ]:
# 13. IDML InDesign Export
# -----------------------------------------------------------------------
# Demonstrates IDMLExporter: building an IDML file from scratch with
# text frames, tables, and image placeholders.
# -----------------------------------------------------------------------

import tempfile
import zipfile

try:
    from siege_utilities.reporting.idml_export import (
        IDMLExporter,
        export_report_idml,
        SIMPLEIDML_AVAILABLE,
    )
    HAS_IDML = True
except ImportError as e:
    HAS_IDML = False
    su.log_info(f"idml_export import failed ({e}) — skipping IDML demo.")

if HAS_IDML:
    su.log_info(f"simpleidml available: {SIMPLEIDML_AVAILABLE}")
    su.log_info("  (Manual ZIP builder works without it.)")

    # --- Build a report IDML from scratch ---
    exporter = IDMLExporter(page_width=612, page_height=792)

    exporter.add_text_frame(
        "Campaign Finance Report", style_name="Title",
        x=72, y=72, width=468, height=36,
    )

    exporter.add_text_frame(
        "Q4 2025 Summary", style_name="Subtitle",
        x=72, y=114, width=468, height=24,
    )

    exporter.add_table(
        headers=["Candidate", "Raised", "Spent", "COH"],
        rows=[
            ["Smith, J.", "$1,250,000", "$980,000", "$270,000"],
            ["Jones, A.", "$890,000", "$750,000", "$140,000"],
            ["Williams, R.", "$2,100,000", "$1,800,000", "$300,000"],
        ],
        x=72, y=160, width=468, height=100,
    )

    exporter.add_image_frame(
        "/path/to/spending_chart.svg",
        x=72, y=280, width=468, height=300,
    )

    with tempfile.TemporaryDirectory() as tmpdir:
        output = exporter.save(f"{tmpdir}/campaign_report")
        output_path = Path(output)
        su.log_info(f"IDML file: {output_path.name}")
        su.log_info(f"  Size: {output_path.stat().st_size:,} bytes")

        with zipfile.ZipFile(output) as zf:
            su.log_info(f"  ZIP entries: {len(zf.namelist())}")
            for name in sorted(zf.namelist()):
                info = zf.getinfo(name)
                su.log_info(f"    {name:<45} {info.file_size:>6} bytes")

In [ ]:
# 13b. export_report_idml() — convenience function for structured data
# -----------------------------------------------------------------------
# Given a dictionary of analytics data (e.g., GA report), produces a
# complete IDML file with text frames and tables laid out automatically.
# -----------------------------------------------------------------------

if HAS_IDML:
    sample_report_data = {
        "date_range": {"start": "2025-10-01", "end": "2025-12-31"},
        "totals": {
            "users": 45200, "sessions": 62800,
            "pageviews": 198500, "avg_bounce_rate": 42.3,
        },
        "changes": {},
        "traffic_sources": [
            {"source": "google", "medium": "organic",
             "sessions": 28000, "users": 22000, "bounce_rate": 38.5},
            {"source": "direct", "medium": "none",
             "sessions": 15000, "users": 12000, "bounce_rate": 45.2},
        ],
        "top_pages": [
            {"page": "/", "pageviews": 52000, "unique_views": 41000,
             "avg_time": 65.3, "bounce_rate": 35.0},
            {"page": "/donate", "pageviews": 18000, "unique_views": 15000,
             "avg_time": 120.5, "bounce_rate": 22.0},
        ],
        "devices": [
            {"device": "mobile", "sessions": 35000, "bounce_rate": 48.0},
            {"device": "desktop", "sessions": 24000, "bounce_rate": 35.0},
        ],
    }

    with tempfile.TemporaryDirectory() as tmpdir:
        result = export_report_idml(
            sample_report_data,
            output_path=f"{tmpdir}/analytics_report",
            title="Analytics Report",
        )
        result_path = Path(result)
        su.log_info(
            f"Analytics IDML: {result_path.name} "
            f"({result_path.stat().st_size:,} bytes)"
        )

---

## Part 2: Advanced ReportLab PDF features

Beyond the one-call PDF exports from Part 1 — direct ReportLab flowable assembly for fine-grained control over page layout, custom frames, and mixed-content sections.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
import tempfile
from IPython.display import display, HTML

# Ensure siege_utilities is importable
sys.path.insert(0, str(Path.cwd().parent))

import siege_utilities as su
su.configure_shared_logging(level="INFO")

print(f"Python: {sys.version}")
print("ReportLab PDF Features notebook ready.")

In [ ]:
# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_11"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Branding configuration ---
from siege_utilities.reporting.client_branding import ClientBrandingManager
_branding_mgr = ClientBrandingManager()
BRANDING = _branding_mgr.get_client_branding('siege_analytics')
COLORS = BRANDING['colors']

print(f"Output directory: {OUTPUT_DIR}")
print(f"Branding: {BRANDING['name']} (primary={COLORS['primary']})")

In [ ]:
# Check ReportLab availability
try:
    from reportlab.platypus import SimpleDocTemplate, Paragraph
    from reportlab.lib.pagesizes import letter
    print("ReportLab: available")
except ImportError as e:
    print(f"ReportLab: NOT available ({e})")

try:
    from PIL import Image
    print("PIL (Pillow): available")
except ImportError as e:
    print(f"PIL: NOT available ({e})")

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    print(f"Matplotlib: available (backend: {matplotlib.get_backend()})")
except ImportError as e:
    print(f"Matplotlib: NOT available ({e})")

In [ ]:
# --- Branded PDF styles & helpers (matches NB06/NB10 patterns) ---
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    PageBreak, HRFlowable,
)
from reportlab.platypus import Image as RLImage
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER, TA_LEFT

# --- ReportLab color objects from branding ---
sa_primary = colors.HexColor(COLORS['primary'])        # #1D365D navy
sa_secondary = colors.HexColor(COLORS['secondary'])    # #4A90E2 blue
sa_accent = colors.HexColor(COLORS['accent'])          # #FF6B35 orange
sa_light_bg = colors.HexColor('#F0F4F8')

# --- ParagraphStyles ---
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(
    'SA_Title', parent=styles['Title'],
    fontSize=32, textColor=sa_primary, spaceAfter=4,
))
styles.add(ParagraphStyle(
    'SA_Subtitle', parent=styles['Normal'],
    fontSize=16, textColor=sa_secondary, alignment=TA_CENTER,
    spaceBefore=8, spaceAfter=20,
))
styles.add(ParagraphStyle(
    'SA_Heading', parent=styles['Heading1'],
    fontSize=18, textColor=sa_primary, spaceBefore=12, spaceAfter=8,
))
styles.add(ParagraphStyle(
    'SA_SubHeading', parent=styles['Heading2'],
    fontSize=14, textColor=sa_secondary, spaceBefore=8, spaceAfter=6,
))
styles.add(ParagraphStyle(
    'SA_Body', parent=styles['Normal'],
    fontSize=11, textColor=colors.HexColor('#333333'), leading=15, spaceAfter=4,
))
styles.add(ParagraphStyle(
    'SA_Bullet', parent=styles['Normal'],
    fontSize=11, textColor=colors.HexColor('#333333'), leading=15,
    leftIndent=20, spaceAfter=2,
))
styles.add(ParagraphStyle(
    'SA_Meta', parent=styles['Normal'],
    fontSize=10, textColor=colors.HexColor('#666666'), alignment=TA_CENTER,
))
styles.add(ParagraphStyle(
    'SA_Caption', parent=styles['Normal'],
    fontSize=9, textColor=colors.HexColor('#888888'), italic=True,
    alignment=TA_CENTER, spaceBefore=4, spaceAfter=8,
))


def make_branded_table(df, col_widths=None):
    """Create a Table with branded header and alternating row backgrounds."""
    data = [df.columns.tolist()] + df.values.tolist()
    tbl = Table(data, colWidths=col_widths)
    tbl.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), sa_primary),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 11),
        ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 1), (-1, -1), 10),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BOTTOMPADDING', (0, 0), (-1, 0), 10),
        ('TOPPADDING', (0, 0), (-1, 0), 8),
        ('BOTTOMPADDING', (0, 1), (-1, -1), 6),
        ('TOPPADDING', (0, 1), (-1, -1), 6),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#CCCCCC')),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, sa_light_bg]),
    ]))
    return tbl


def branded_title_page(story, title, subtitle, meta_lines=None):
    """Append a branded title page to story."""
    story.append(Spacer(1, 2.0 * inch))
    story.append(HRFlowable(width='70%', thickness=3, color=sa_primary, spaceAfter=16))
    story.append(Paragraph(title, styles['SA_Title']))
    story.append(Paragraph(subtitle, styles['SA_Subtitle']))
    story.append(HRFlowable(width='70%', thickness=1, color=sa_secondary, spaceBefore=4, spaceAfter=30))
    for line in (meta_lines or []):
        story.append(Paragraph(line, styles['SA_Meta']))
        story.append(Spacer(1, 4))
    story.append(PageBreak())


def section_header(story, title):
    """Append a branded section heading with rule."""
    story.append(HRFlowable(width='100%', thickness=2, color=sa_primary, spaceAfter=8))
    story.append(Paragraph(title, styles['SA_Heading']))
    story.append(Spacer(1, 4))


print("Branded PDF styles ready: SA_Title, SA_Subtitle, SA_Heading, SA_SubHeading,")
print("  SA_Body, SA_Bullet, SA_Meta, SA_Caption")
print("Helpers: make_branded_table(), branded_title_page(), section_header()")

## 1. Basic PDF Generation

Test that basic PDF generation works.

In [ ]:
from siege_utilities.reporting.report_generator import ReportGenerator

# Initialize ReportGenerator — output goes to OUTPUT_DIR
rg = ReportGenerator(
    client_name="TestClient",
    output_dir=OUTPUT_DIR
)

print(f"ReportGenerator initialized (output: {OUTPUT_DIR})")

In [ ]:
# Create sample data
np.random.seed(42)

# Quarterly sales data
sales_data = pd.DataFrame({
    'Quarter': ['Q1 2025', 'Q2 2025', 'Q3 2025', 'Q4 2025'],
    'Revenue ($M)': [12.5, 15.2, 18.7, 22.1],
    'Growth (%)': [8.5, 12.1, 23.0, 18.2],
    'New Customers': [145, 189, 234, 278]
})

# Regional breakdown
regional_data = pd.DataFrame({
    'Region': ['Northeast', 'Southeast', 'Midwest', 'West', 'International'],
    'Revenue ($M)': [18.2, 12.5, 8.7, 22.4, 6.7],
    'Market Share (%)': [27, 18, 13, 33, 10]
})

print("Sales Data:")
display(sales_data)
print("\nRegional Data:")
display(regional_data)

In [ ]:
# Build basic report content
report_content = {
    'sections': [],
    'metadata': {
        'title': 'Quarterly Business Report',
        'subtitle': 'FY 2025 Performance Analysis',
        'author': 'Siege Analytics',
        'date': datetime.now().strftime('%B %d, %Y'),
        'client': 'TestClient'
    }
}

exec_summary = """
This report presents the quarterly performance analysis for FY 2025.
Key highlights include:

- Total revenue of $68.5M, up 15.4% year-over-year
- Strong growth in Q3 with 23% quarter-over-quarter increase
- 846 new customers acquired across all regions
- West region leads with 33% market share

The company is well-positioned for continued growth in FY 2026.
"""

report_content = rg.add_text_section(
    report_content,
    'Executive Summary',
    exec_summary
)

print("Report content built with executive summary.")

In [ ]:
# Add data tables
report_content = rg.add_table_section(report_content, 'Quarterly Performance', sales_data)
report_content = rg.add_table_section(report_content, 'Regional Breakdown', regional_data)

print(f"Report now has {len(report_content['sections'])} sections.")

In [ ]:
# Generate branded basic PDF
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
basic_pdf_path = OUTPUT_DIR / f"basic_report_{timestamp}.pdf"

try:
    story = []
    meta = report_content['metadata']

    # Branded title page
    branded_title_page(
        story,
        title=meta['title'],
        subtitle=meta['subtitle'],
        meta_lines=[
            f"Prepared by: {meta.get('author', 'Siege Analytics')}",
            f"Date: {meta.get('date', '')}",
        ],
    )

    # Render each section with branded styles
    for sec in report_content['sections']:
        section_header(story, sec['title'])
        if sec['type'] == 'text':
            for line in sec['content'].strip().split('\n'):
                line = line.strip()
                if not line:
                    story.append(Spacer(1, 6))
                elif line.startswith('- '):
                    story.append(Paragraph(f"\u2022 {line[2:]}", styles['SA_Bullet']))
                else:
                    story.append(Paragraph(line, styles['SA_Body']))
            story.append(Spacer(1, 12))
        elif sec['type'] == 'table':
            # add_table_section stores content as {'data': [[hdr,...], [row,...], ...]}
            table_rows = sec['content']['data']
            df = pd.DataFrame(table_rows[1:], columns=table_rows[0])
            story.append(make_branded_table(df))
            story.append(Spacer(1, 16))

    doc = SimpleDocTemplate(
        str(basic_pdf_path), pagesize=letter,
        topMargin=1*inch, bottomMargin=1*inch,
        leftMargin=1*inch, rightMargin=1*inch,
    )
    doc.build(story)

    if basic_pdf_path.exists():
        size_kb = basic_pdf_path.stat().st_size / 1024
        print(f"Basic PDF generated: {basic_pdf_path.name} ({size_kb:.1f} KB)")
    else:
        print("WARNING: PDF was not created")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback
    traceback.print_exc()

## 2. Multi-Page PDF with Page Breaks

Test multi-page PDF generation with explicit page breaks.

In [ ]:
# Create multi-page report
multipage_content = {
    'sections': [],
    'metadata': {
        'title': 'Comprehensive Annual Report',
        'subtitle': 'Multi-Section Analysis',
        'author': 'Siege Analytics',
        'date': datetime.now().strftime('%B %d, %Y'),
    }
}

# Section 1: Introduction (with page break after)
intro_text = """
This comprehensive report covers multiple aspects of business performance.
The analysis spans financial metrics, market position, customer analysis,
and strategic recommendations.

Report Structure:
1. Financial Performance
2. Market Analysis
3. Customer Insights
4. Strategic Recommendations
"""
multipage_content = rg.add_section(
    multipage_content, 'text', 'Introduction', intro_text, page_break_after=True
)

# Section 2: Financial Performance
financial_text = """
The financial performance in FY 2025 exceeded expectations across all key metrics.
Revenue grew by 15.4% compared to FY 2024, driven primarily by expansion
in the West region and successful product launches.

Key Financial Highlights:
- Total Revenue: $68.5M
- Gross Margin: 72.3%
- Operating Income: $18.2M
- Free Cash Flow: $12.8M
"""
multipage_content = rg.add_text_section(multipage_content, 'Financial Performance', financial_text)
multipage_content = rg.add_table_section(multipage_content, 'Quarterly Financials', sales_data)
multipage_content['sections'][-1]['page_break_after'] = True

# Section 3: Market Analysis
market_text = """
Market analysis reveals strong competitive positioning across all regions.
The company has successfully expanded market share in key territories
while maintaining premium pricing power.
"""
multipage_content = rg.add_text_section(multipage_content, 'Market Analysis', market_text)
multipage_content = rg.add_table_section(multipage_content, 'Regional Market Share', regional_data)

# Section 4: Recommendations
recommendations_text = """
Strategic priorities for FY 2026:

1. Continue investment in West region growth initiatives
2. Expand international presence, targeting European markets
3. Launch customer retention program to reduce churn
4. Increase R&D spending to maintain technology leadership
5. Explore strategic acquisitions in adjacent markets
"""
multipage_content = rg.add_text_section(multipage_content, 'Strategic Recommendations', recommendations_text)

print(f"Multi-page report built with {len(multipage_content['sections'])} sections.")

In [ ]:
# Generate branded multi-page PDF
multipage_pdf_path = OUTPUT_DIR / f"multipage_report_{timestamp}.pdf"

try:
    story = []
    meta = multipage_content['metadata']

    branded_title_page(
        story,
        title=meta['title'],
        subtitle=meta['subtitle'],
        meta_lines=[
            f"Prepared by: {meta.get('author', 'Siege Analytics')}",
            f"Date: {meta.get('date', '')}",
        ],
    )

    for sec in multipage_content['sections']:
        section_header(story, sec['title'])
        if sec['type'] == 'text':
            for line in sec['content'].strip().split('\n'):
                line = line.strip()
                if not line:
                    story.append(Spacer(1, 6))
                elif line.startswith('- '):
                    story.append(Paragraph(f"\u2022 {line[2:]}", styles['SA_Bullet']))
                else:
                    story.append(Paragraph(line, styles['SA_Body']))
            story.append(Spacer(1, 12))
        elif sec['type'] == 'table':
            # add_table_section stores content as {'data': [[hdr,...], [row,...], ...]}
            table_rows = sec['content']['data']
            df = pd.DataFrame(table_rows[1:], columns=table_rows[0])
            story.append(make_branded_table(df))
            story.append(Spacer(1, 16))

        if sec.get('page_break_after'):
            story.append(PageBreak())

    doc = SimpleDocTemplate(
        str(multipage_pdf_path), pagesize=letter,
        topMargin=1*inch, bottomMargin=1*inch,
        leftMargin=1*inch, rightMargin=1*inch,
    )
    doc.build(story)

    if multipage_pdf_path.exists():
        size_kb = multipage_pdf_path.stat().st_size / 1024
        print(f"Multi-page PDF generated: {multipage_pdf_path.name} ({size_kb:.1f} KB)")
    else:
        print("WARNING: PDF was not created")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback
    traceback.print_exc()

## 3. Table of Contents Template

Test the dedicated Table of Contents template.

In [ ]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from siege_utilities.reporting.templates.table_of_contents_template import (
    TableOfContentsTemplate,
    create_table_of_contents
)

# Define TOC sections
toc_sections = [
    {
        'title': 'Executive Summary', 'page': 3,
        'subsections': [
            {'title': 'Key Findings', 'page': 3},
            {'title': 'Recommendations', 'page': 4}
        ]
    },
    {
        'title': 'Financial Performance', 'page': 5,
        'subsections': [
            {'title': 'Revenue Analysis', 'page': 5},
            {'title': 'Cost Structure', 'page': 6},
            {'title': 'Profitability Metrics', 'page': 7}
        ]
    },
    {
        'title': 'Market Analysis', 'page': 8,
        'subsections': [
            {'title': 'Competitive Landscape', 'page': 8},
            {'title': 'Market Share Trends', 'page': 9}
        ]
    },
    {'title': 'Strategic Recommendations', 'page': 10},
    {'title': 'Appendices', 'page': 11}
]

print(f"TOC defined: {len(toc_sections)} main sections")

In [ ]:
# Generate TOC PDF
toc_pdf_path = OUTPUT_DIR / f"table_of_contents_{timestamp}.pdf"

try:
    c = canvas.Canvas(str(toc_pdf_path), pagesize=letter)
    create_table_of_contents(
        canvas_obj=c, sections=toc_sections,
        title="Table of Contents", page_number=2
    )
    c.save()
    
    if toc_pdf_path.exists():
        size_kb = toc_pdf_path.stat().st_size / 1024
        print(f"TOC PDF generated: {toc_pdf_path.name} ({size_kb:.1f} KB)")
    else:
        print("WARNING: PDF was not created")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback
    traceback.print_exc()

## 4. Charts in Reports

Test embedding matplotlib charts in PDF reports.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Create sample charts using branding colors
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart: Revenue by quarter
quarters = sales_data['Quarter']
revenue = sales_data['Revenue ($M)']
bar_colors = [COLORS['primary'], COLORS['secondary'], COLORS.get('accent', '#FF6B35'), COLORS.get('background', '#F5F5F5')]

axes[0].bar(quarters, revenue, color=bar_colors)
axes[0].set_title('Quarterly Revenue', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Revenue ($M)')
axes[0].set_xlabel('Quarter')
for i, v in enumerate(revenue):
    axes[0].text(i, v + 0.5, f'${v}M', ha='center', fontsize=10)

# Pie chart: Regional market share
regions = regional_data['Region']
market_share = regional_data['Market Share (%)']
pie_colors = [COLORS['primary'], COLORS['secondary'], COLORS.get('accent', '#FF6B35'), COLORS.get('text_color', '#333333'), COLORS.get('header_footer_text_color', '#FFFFFF')]

axes[1].pie(market_share, labels=regions, autopct='%1.0f%%', colors=pie_colors, startangle=90)
axes[1].set_title('Regional Market Share', fontsize=14, fontweight='bold')

plt.tight_layout()

# Save chart for embedding in PDF
chart_path = OUTPUT_DIR / f"charts_{timestamp}.png"
plt.savefig(chart_path, dpi=150, bbox_inches='tight', facecolor='white')

# Show inline in notebook before closing
plt.show()
plt.close()

print(f"Chart saved: {chart_path.name} ({chart_path.stat().st_size / 1024:.1f} KB)")

In [ ]:
# Create branded report with embedded chart
chart_report_path = OUTPUT_DIR / f"report_with_charts_{timestamp}.pdf"

try:
    story = []

    branded_title_page(
        story,
        title='Business Analytics Report',
        subtitle='With Embedded Charts',
        meta_lines=[
            'Prepared by: Siege Analytics',
            f"Date: {datetime.now().strftime('%B %d, %Y')}",
        ],
    )

    section_header(story, 'Data Visualizations')
    story.append(Paragraph(
        "The following charts illustrate key business metrics including quarterly "
        "revenue trends and regional market share distribution.",
        styles['SA_Body'],
    ))
    story.append(Spacer(1, 24))

    if chart_path.exists():
        img = RLImage(str(chart_path), width=7*inch, height=3.5*inch)
        story.append(img)
        story.append(Spacer(1, 8))
        story.append(Paragraph(
            "Figure 1: Quarterly Revenue and Regional Market Share",
            styles['SA_Caption'],
        ))

    doc = SimpleDocTemplate(
        str(chart_report_path), pagesize=letter,
        topMargin=1*inch, bottomMargin=1*inch,
        leftMargin=1*inch, rightMargin=1*inch,
    )
    doc.build(story)

    if chart_report_path.exists():
        size_kb = chart_report_path.stat().st_size / 1024
        print(f"Chart report PDF generated: {chart_report_path.name} ({size_kb:.1f} KB)")
    else:
        print("WARNING: PDF was not created")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback
    traceback.print_exc()

## 5. Comprehensive Report with All Features

Test a full report combining all features.

In [ ]:
# Branded comprehensive report with all features
comprehensive_report_path = OUTPUT_DIR / f"comprehensive_report_{timestamp}.pdf"

try:
    story = []

    # --- Title page ---
    branded_title_page(
        story,
        title='Annual Business Report',
        subtitle='FY 2025 Comprehensive Analysis',
        meta_lines=[
            'Prepared by: Siege Analytics',
            f"Date: {datetime.now().strftime('%B %d, %Y')}",
        ],
    )

    # --- Executive Summary ---
    section_header(story, 'Executive Summary')
    story.append(Paragraph(
        "This comprehensive report presents the financial and operational performance "
        "for FY 2025. The company achieved strong results across all key metrics, "
        "with revenue growing 15.4% year-over-year to $68.5M.",
        styles['SA_Body'],
    ))
    story.append(Spacer(1, 20))

    # --- Quarterly Performance ---
    section_header(story, 'Quarterly Performance')
    story.append(make_branded_table(sales_data))
    story.append(Spacer(1, 20))

    # --- Data Visualizations ---
    section_header(story, 'Data Visualizations')
    if chart_path.exists():
        img = RLImage(str(chart_path), width=6*inch, height=3*inch)
        story.append(img)
        story.append(Spacer(1, 8))
        story.append(Paragraph(
            "Figure 1: Quarterly Revenue and Regional Market Share",
            styles['SA_Caption'],
        ))
    story.append(PageBreak())

    # --- Regional Analysis ---
    section_header(story, 'Regional Analysis')
    story.append(Paragraph(
        "The regional breakdown shows strong performance across all territories. "
        "The West region continues to lead with 33% market share.",
        styles['SA_Body'],
    ))
    story.append(Spacer(1, 16))
    story.append(make_branded_table(regional_data))
    story.append(Spacer(1, 20))

    # --- Strategic Recommendations ---
    section_header(story, 'Strategic Recommendations')
    for i, rec in enumerate([
        "Continue investment in West region growth initiatives",
        "Expand international presence targeting European markets",
        "Launch customer retention program to reduce churn",
        "Increase R&D spending to maintain technology leadership",
        "Explore strategic acquisitions in adjacent markets",
    ], 1):
        story.append(Paragraph(f"{i}. {rec}", styles['SA_Body']))
        story.append(Spacer(1, 4))

    doc = SimpleDocTemplate(
        str(comprehensive_report_path), pagesize=letter,
        topMargin=1*inch, bottomMargin=1*inch,
        leftMargin=1*inch, rightMargin=1*inch,
    )
    doc.build(story)

    if comprehensive_report_path.exists():
        size_kb = comprehensive_report_path.stat().st_size / 1024
        print(f"Comprehensive PDF generated: {comprehensive_report_path.name} ({size_kb:.1f} KB)")
    else:
        print("WARNING: PDF was not created")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Summary — check all generated files
print("=" * 50)
print("REPORTLAB PDF FEATURE TEST RESULTS")
print("=" * 50)

generated_files = [
    ('Basic PDF', basic_pdf_path),
    ('Multi-page PDF', multipage_pdf_path),
    ('Table of Contents', toc_pdf_path),
    ('Charts Report', chart_report_path),
    ('Comprehensive Report', comprehensive_report_path),
]

results = []
for name, path in generated_files:
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"  PASS  {name}: {size_kb:.1f} KB")
        results.append(True)
    else:
        print(f"  FAIL  {name}: Not generated")
        results.append(False)

passed = sum(results)
total = len(results)
print(f"\n{passed}/{total} PDFs generated successfully.")
print(f"Output directory: {OUTPUT_DIR}")

if passed == total:
    print("\nAll ReportLab PDF features working!")
else:
    print("\nSome features need attention.")

## Related
- Source: `siege_utilities/reporting/` (chart_generator, report_generator, chart_types)
- Tests: `tests/test_chart_types*.py`, `tests/test_reporting_config_exports.py`
- Consolidates: `11_ReportLab_PDF_Features`
